# Lab00 — A clean project for Nova Market

**Goal:** a brand-new Google Cloud project with every API the workshop needs, the Agents CLI
installed, and one `workshop.env` file that all later labs read.

**You will:**
1. Choose names (project, region, model)
2. Create the project and link billing
3. Enable the APIs used across Lab01–Lab07
4. Install the **Agents CLI** (and its coding-assistant skills)
5. Verify you can call **Gemini 3.8 Flash** on the EU endpoint
6. Save everything to `workshop.env`

> **Why a fresh project?** Agent Gateway policies, Model Armor floor settings and IAM access
> policies are project-wide. Starting clean means nothing from an old experiment interferes,
> and clean-up at the end (Lab09) is a single `gcloud projects delete`.

Estimated time: 10 minutes (API enablement is the slow part).

> **Terminal or notebook — your choice.** Every cell that calls a CLI prints the exact command first (`$ …`).
> Copy it into your own terminal if you prefer to run it yourself; the cells just automate the same commands.

## 0.1 Before you start

Make sure `gcloud` is authenticated **twice** — once for the CLI and once for Application
Default Credentials (ADC), which the Python SDKs and the Agents CLI use:

```bash
gcloud auth login
gcloud auth application-default login
```

Run the next cell to confirm.

In [ ]:
# --- Shell helper + check that gcloud is authenticated twice (CLI login and Application Default Credentials) ---
import os, subprocess, json, pathlib, re, datetime
os.environ["CLOUDSDK_CORE_DISABLE_PROMPTS"] = "1"   # never let gcloud wait for a (y/N) answer inside a notebook

def sh(cmd, check=True, quiet=False):
    """Run a shell command, echo it and its output, return stdout; raise on failure unless check=False."""
    if not quiet:
        print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout and not quiet:
        print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip())
        if check:
            raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r.stdout.strip()

# Check the CLI login: this account will create and own the new project.
ACCOUNT = sh("gcloud config get-value account", quiet=True)
print("gcloud account:", ACCOUNT)

# Check Application Default Credentials: the Python SDKs used in the labs authenticate with these, not with the CLI login.
sh("gcloud auth application-default print-access-token >/dev/null && echo 'ADC: OK'")

## 0.2 Choose your names

Edit the values below. The **project ID must be globally unique** (6–30 chars, lowercase,
digits, hyphens); the default appends today's date.

| Variable | Meaning |
| --- | --- |
| `PROJECT_ID` | the new project |
| `BILLING_ACCOUNT` | leave empty to pick the first open billing account you can see |
| `PARENT` | `organizations/123…` or `folders/456…` — or empty for no parent |
| `REGION` | `europe-west1` (Belgium): Agent Runtime, Sessions, Memory Bank, Agent Gateway, Model Armor |
| `MODEL_LOCATION` | `eu`: the multi-region endpoint that serves Gemini 3.8 Flash inside the EU |
| `MODEL` | `gemini-3.8-flash` |

In [ ]:
# --- Workshop names: edit before running (the project id must be globally unique) ---
PROJECT_ID      = f"my-geap-workshop-{datetime.date.today():%y%m%d}"   # <-- change if you like; must be globally unique
BILLING_ACCOUNT = ""                                              # e.g. "012345-6789AB-CDEF01"; empty = auto-pick
PARENT          = ""                                              # e.g. "organizations/752235872408"; empty = auto-detect / none
REGION          = "europe-west1"                                  # Agent Runtime, Sessions, Memory Bank, Gateway, Model Armor
MODEL_LOCATION  = "eu"                                            # multi-region model endpoint that serves gemini-3.8-flash
MODEL           = "gemini-3.8-flash"
AGENT_NAME      = "nova-assistant"                                # name of the agent project and of the deployed agent

In [ ]:
# --- Validate the project id -> pick a billing account -> pick a parent organization ---
# Check the project id format before spending any API calls on it.
assert re.fullmatch(r"[a-z][a-z0-9-]{4,28}[a-z0-9]", PROJECT_ID), "invalid project id"

# Pick the billing account: the first open one you can see, unless you set BILLING_ACCOUNT above.
if not BILLING_ACCOUNT:
    accounts = json.loads(sh("gcloud billing accounts list --filter=open=true --format=json", quiet=True))
    assert accounts, "No open billing account visible - set BILLING_ACCOUNT manually"
    BILLING_ACCOUNT = accounts[0]["name"].split("/")[-1]
    print("Using billing account:", BILLING_ACCOUNT, "-", accounts[0]["displayName"])

# Pick the parent: your organization if there is exactly one; otherwise the project is created without one or you set PARENT.
if not PARENT:
    orgs = json.loads(sh("gcloud organizations list --format=json", quiet=True) or "[]")
    if len(orgs) == 1:
        PARENT = orgs[0]["name"]          # organizations/NNN
        print("Using organization:", PARENT, "-", orgs[0]["displayName"])
    else:
        print("No single organization detected; project will be created without a parent" if not orgs else f"Several organizations visible - set PARENT explicitly: {[o['name'] for o in orgs]}")

## 0.3 Create the project and link billing

The cell is idempotent: if the project already exists it is reused.

In [ ]:
# --- Create the project (or reuse it) -> link billing -> read the project number ---
# Create the project only if it does not exist yet, so re-running this cell is safe.
existing = sh(f"gcloud projects list --filter='projectId={PROJECT_ID}' --format='value(projectId)'", quiet=True)
if existing:
    print(f"Project {PROJECT_ID} already exists - reusing it")
else:
    # gcloud wants the parent as --organization or --folder, so translate the PARENT string.
    parent_flag = ""
    if PARENT.startswith("organizations/"):
        parent_flag = f"--organization={PARENT.split('/')[1]}"
    elif PARENT.startswith("folders/"):
        parent_flag = f"--folder={PARENT.split('/')[1]}"
    sh(f"gcloud projects create {PROJECT_ID} --name='GEAP e-shop workshop' {parent_flag}")

# Link billing: without it no API can be enabled.
sh(f"gcloud billing projects link {PROJECT_ID} --billing-account={BILLING_ACCOUNT}")

# Read the project number: service-account and agent-identity principals are built from it in later labs.
PROJECT_NUMBER = sh(f"gcloud projects describe {PROJECT_ID} --format='value(projectNumber)'", quiet=True)
print("Project number:", PROJECT_NUMBER)

## 0.4 Point `gcloud` at the new project

We create a **named gcloud configuration** so the workshop never touches your default
project. Activate it in any terminal with `gcloud config configurations activate geap-workshop`.

In [ ]:
# --- Point gcloud at the new project through its own configuration "geap-workshop" ---
# Create the configuration once (the first command fails harmlessly if it already exists), then activate it.
sh("gcloud config configurations describe geap-workshop >/dev/null 2>&1 || gcloud config configurations create geap-workshop", check=False)
sh("gcloud config configurations activate geap-workshop")

# Set account, project and default region inside that configuration only; your default gcloud config stays untouched.
sh(f"gcloud config set account {ACCOUNT}")
sh(f"gcloud config set project {PROJECT_ID}")
sh(f"gcloud config set compute/region {REGION}")

# Make the SDKs and shell commands in this kernel target the same project.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

## 0.5 Enable the APIs

One list for the whole workshop, grouped by the lab that first needs each API.
(`gcloud services enable` accepts at most 20 services per call, hence the batches.)

| Lab | APIs |
| --- | --- |
| all | Service Usage, Resource Manager, IAM, IAM Credentials, Org Policy |
| 01–02 | **Agent Platform** (`aiplatform`) — models, Agent Runtime, Sessions, Memory Bank, Code Execution, evaluation; Cloud Storage, Cloud Build, Artifact Registry (container builds); **Agent Registry**; **Discovery Engine** (Gemini Enterprise) |
| 03 | BigQuery |
| 04 | Cloud Run (mock systems), Compute, Network Services + Network Security (**Agent Gateway**), Cloud DNS, **IAP** (policy enforcement), Secret Manager |
| 05 | **Model Armor**, Sensitive Data Protection (`dlp`) |
| 06 | Logging, Monitoring, Cloud Trace, Telemetry, Observability, App Hub, App Topology, Cloud API Registry |

In [ ]:
# --- Enable every API the workshop needs (one list, grouped by the lab that first uses it) ---
APIS = [
    # platform basics
    "serviceusage.googleapis.com", "cloudresourcemanager.googleapis.com", "iam.googleapis.com",
    "iamcredentials.googleapis.com", "orgpolicy.googleapis.com",
    # Lab01-02: Agent Platform, builds, registry, Gemini Enterprise
    "aiplatform.googleapis.com", "storage.googleapis.com", "cloudbuild.googleapis.com",
    "artifactregistry.googleapis.com", "agentregistry.googleapis.com", "discoveryengine.googleapis.com",
    # Lab03: data and the mock enterprise systems on Cloud Run
    "bigquery.googleapis.com", "run.googleapis.com", "compute.googleapis.com",
    # Lab04: observability
    "logging.googleapis.com", "monitoring.googleapis.com", "cloudtrace.googleapis.com",
    "telemetry.googleapis.com", "observability.googleapis.com", "apphub.googleapis.com",
    "apptopology.googleapis.com", "cloudapiregistry.googleapis.com",
    # Lab05: Model Armor
    "modelarmor.googleapis.com", "dlp.googleapis.com",
    # Lab06: gateway, identity, policies
    "networkservices.googleapis.com", "networksecurity.googleapis.com", "dns.googleapis.com",
    "iap.googleapis.com", "secretmanager.googleapis.com",
]

# Enable them in batches: gcloud accepts at most 20 services per call.
for i in range(0, len(APIS), 20):
    sh(f"gcloud services enable {' '.join(APIS[i:i+20])} --project={PROJECT_ID}")

# Verify: compare the enabled list with ours and report anything missing.
enabled = set(sh(f"gcloud services list --enabled --project={PROJECT_ID} --format='value(config.name)'", quiet=True).split())
missing = [a for a in APIS if a not in enabled]
print("missing:", missing or "none - all APIs enabled")

# Set the ADC quota project (possible now that Resource Manager is enabled): SDK calls from this
# notebook are then billed and quota-checked against this project, not against some other one.
sh(f"gcloud auth application-default set-quota-project {PROJECT_ID} --quiet", check=False)

## 0.6 Install the Agents CLI

The **Agents CLI** (`agents-cli`) wraps the ADK, the evaluation SDK and the deployment
tooling into one command line: `create → playground → eval → deploy → publish`.
It also installs seven *skills* into whatever coding assistant it detects (Antigravity,
Gemini CLI, Codex, Cursor…). Those skills contain the same guidance you'll
follow by hand in these labs — so after the workshop you can say
*"use agents-cli to add a returns tool and re-run the evals"* and let your assistant do it.

`setup` is the only command you ever run through `uvx`; afterwards `agents-cli` is on your PATH.

In [ ]:
# --- Install the Agents CLI and check that it is logged in ---
# One-time setup: installs agents-cli and adds its skills to the coding assistants it detects on this machine.
sh("uvx google-agents-cli setup --skip-auth", check=False)
print()

# Confirm the version and the login status.
sh("agents-cli --version")
sh("agents-cli login --status", check=False)

## 0.7 Verify model access on the EU endpoint

Gemini 3.8 Flash is served from the `global`, `us` and `eu` endpoints (not from single
regions such as `europe-west1`). The ADK and the GenAI SDK pick the endpoint from
`GOOGLE_CLOUD_LOCATION`, so the workshop sets that to **`eu`** everywhere, while the agent
itself runs in **`europe-west1`**. Two different knobs — keep them apart in your head:

* `GOOGLE_CLOUD_LOCATION=eu` → where the **model** is called
* `--region europe-west1` → where the **agent** (runtime, sessions, memory, gateway) lives

In [ ]:
# --- Verify model access: one Gemini call on the eu endpoint, retried while a fresh project's IAM propagates ---
import time
from google import genai
# The client has to live in a variable: a temporary one is closed before the request is sent.
client = genai.Client(vertexai=True, project=PROJECT_ID, location=MODEL_LOCATION)

# Call the model. A brand-new project can take 1-3 minutes to propagate IAM + API enablement, so retry on 403.
for attempt in range(12):
    try:
        resp = client.models.generate_content(model=MODEL, contents="In one sentence: what is Nova Market? (It is a fictional EU electronics marketplace.)")
        print(resp.text)
        break
    except Exception as e:
        if "PERMISSION_DENIED" in str(e) or "403" in str(e):
            print(f"not ready yet (attempt {attempt+1}): permissions still propagating, retrying in 20s")
            time.sleep(20)
        else:
            raise
else:
    raise RuntimeError("Model call still denied after 4 minutes - check IAM (you need roles/aiplatform.user or Owner) and that aiplatform.googleapis.com is enabled")

## 0.8 Save `workshop.env`

Every other lab starts by loading this file.

In [ ]:
# --- Save workshop.env: every other lab starts by loading this file ---
# Resolve the repo root whether the kernel runs in labs/ or in the repo root.
REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "labs" else pathlib.Path.cwd()
ENV_FILE = REPO_ROOT / "workshop.env"

# Write the values; later labs append their own (agent ids, dataset names) to the same file.
ENV_FILE.write_text(
    f"PROJECT_ID={PROJECT_ID}\n"
    f"PROJECT_NUMBER={PROJECT_NUMBER}\n"
    f"REGION={REGION}\n"
    f"MODEL_LOCATION={MODEL_LOCATION}\n"
    f"MODEL={MODEL}\n"
    f"AGENT_NAME={AGENT_NAME}\n"
)
print(ENV_FILE.read_text())

## Recap

* You have a clean project with every API for Lab01–Lab07 enabled.
* `gcloud` uses the `geap-workshop` configuration; ADC has the project as quota project.
* `agents-cli` is installed and Gemini 3.8 Flash answers from the `eu` endpoint.
* `workshop.env` carries the settings forward.

**Next:** [Lab01 — build and test the Nova Assistant locally](lab01_local_agent.ipynb).